
GPIO pins: 7, 12, 15, 35, 38, 40 


In [6]:
import os
import sys

# Explicit Jetson project root
PROJECT_ROOT = "/home/jpobar/egh490_vpp_project"

# Make project folders importable
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Check where Python is looking
print("Working directory:", os.getcwd())
print("Project root added:", PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

from hardware.hardware_interface import HardwareInterface
print("HardwareInterface imported successfully")

# ============================================================
# PROFILE DATA
# ============================================================

summer_data = pd.read_csv(
    "SUMMER Generation and Load profiles.csv"
)

winter_data = pd.read_csv(
    "WINTER Generation and Load profiles.csv"
)


print("\n=== SUMMER PROFILE DATA CHECK ===")
print(f"Rows loaded: {len(summer_data)}")
print(f"Columns loaded: {len(summer_data.columns)}")
print(f"Missing values: {summer_data.isna().sum().sum()}")

print("\nFirst 5 rows:")
print(summer_data.head())

print("\nColumn names:")
print(summer_data.columns.tolist())


Working directory: /home/jpobar/egh490_vpp_project/Notebooks
Project root added: /home/jpobar/egh490_vpp_project
HardwareInterface imported successfully

=== SUMMER PROFILE DATA CHECK ===
Rows loaded: 336
Columns loaded: 10
Missing values: 0

First 5 rows:
  LOAD_SOURCE_TIMESTEP PV_SOURCE_TIMESTAMP  PV_6.6kW_kW  PV_13.3kW_kW  \
0      04/02/2013 0:00     07/01/2013 0:00          0.0           0.0   
1      04/02/2013 0:30     07/01/2013 0:30          0.0           0.0   
2      04/02/2013 1:00     07/01/2013 1:00          0.0           0.0   
3      04/02/2013 1:30     07/01/2013 1:30          0.0           0.0   
4      04/02/2013 2:00     07/01/2013 2:00          0.0           0.0   

   PV_20kW_kW  H1_HI_B_kW  H2_MED_B_kW  H3_LOW_A_Kw  H4_Hi_A_kW  H5_Med_A_kW  
0         0.0       0.706        0.150        0.135       0.213        0.353  
1         0.0       0.695        0.154        0.136       0.216        0.390  
2         0.0       0.646        0.177        0.059       0.202    

Define simulation Objects


In [13]:
class VPPSimulation:
    def __init__(self,
                 houses,
                 hospital,
                 wind,
                 controller,
                 timestep_hours,
                 num_steps
                ):
        
        self.houses = houses
        self.hospital = hospital
        self.wind = wind
        self.controller = controller

        self.timestep_hours = timestep_hours
        self.num_steps = num_steps

        self.current_step = 0

        self.simulation_results_summary = []

        self.house_results_log = {
            house.name: [] for house in houses
        }
    def step(self, external_inputs=None):
        if external_inputs is None:
            
            external_inputs = {}
            
        profile_index = self.current_step % self.num_steps

        # Clock time used by scheduled controller actions.
        # A 0.5 h timestep gives:
        # step 0 = 00:00, step 1 = 00:30, ... step 32 = 16:00
        hour_of_day = (
            self.current_step * self.timestep_hours
        ) % 24


        # A10 - Update effective export limits before the controller
        # evaluates available VPP capability.
        network_support_request = external_inputs.get(
            "network_support_request",
            False
        )
        
        for house in self.houses:
            house.update_export_limit(network_support_request)
        
        

        house_states = [
            house.get_state(
                profile_index,
                self.timestep_hours
                )
                for house in self.houses
        ]

        hospital_result = self.hospital.step(profile_index)
        wind_result = self.wind.step(profile_index)
                
        signals = self.controller.get_signals(
            hour_of_day,
            external_inputs,
            house_states,
            hospital_result,
            wind_result
        )
            

        house_results = [
            house.step(
                profile_index,
                self.timestep_hours,
                signals
            )
            for house in self.houses
        ]

                # Aggregate results from all houses
        total_house_demand_kw = sum(
            h["load_kw"] for h in house_results
        )

        total_house_generation_kw = sum(
            h["generation_kw"] for h in house_results
        )

        total_house_net_power_kw = sum(
            h["net_power_kw"] for h in house_results
        )

        total_house_net_power_before_battery_kw = sum(
            h["net_power_kw_before_battery"] for h in house_results
        )

        total_battery_power_kw = sum(
            h["battery_power_kw"] for h in house_results
        )

        total_battery_soc_kwh = sum(
            h["battery_soc_kwh"] for h in house_results
        )

        total_battery_discharge_kw = sum(
            h["battery_discharge_kw"] for h in house_results
        )

        total_battery_charge_kw = sum(
            h["battery_charge_kw"] for h in house_results
        )

        total_local_load_kw = hospital_result["load_kw"]
        total_wind_generation_kw = wind_result["generation_kw"]


        # ---------------------------------------------------------
        # VPP grid exchange
        # ---------------------------------------------------------
        # The VPP consists only of the participating houses
        # and their DERs.
        
        vpp_net_power_kw = total_house_net_power_kw
        
        vpp_grid_import_kw = max(
            vpp_net_power_kw,
            0.0
        )
        
        vpp_grid_export_kw = -min(
            vpp_net_power_kw,
            0.0
        )
        
        
        # ---------------------------------------------------------
        # Wider demonstration/network power balance
        # ---------------------------------------------------------
        # Hospital and wind/DFIG exist outside the VPP.
        #
        # They are included here only to represent the wider
        # demonstration environment.
        
        network_net_power_kw = (
            vpp_net_power_kw
            + total_local_load_kw
            - total_wind_generation_kw
        )
        
        network_grid_import_kw = max(
            network_net_power_kw,
            0.0
        )
        
        network_grid_export_kw = -min(
            network_net_power_kw,
            0.0
        )

        # Calculate simulation time
        elapsed_hours = self.current_step * self.timestep_hours
        day_number = int(elapsed_hours // 24) + 1
        


        # Build result for this simulation step
        timestep_result = {
            "Simulation_step": self.current_step,
            "Profile_index": profile_index,
            "Elapsed_hours": elapsed_hours,
            "Day_number": day_number,
            "Hour_of_day": hour_of_day,

            "House_demand_kw": total_house_demand_kw,
            "House_generation_kw": total_house_generation_kw,
            "House_net_power_before_battery_kw":
                total_house_net_power_before_battery_kw,

            "Battery_power_kw": total_battery_power_kw,
            "Battery_soc_kwh": total_battery_soc_kwh,
            "Battery_discharge_kw": total_battery_discharge_kw,
            "Battery_charge_kw": total_battery_charge_kw,
            
            "Battery_discharge_kwh": total_battery_discharge_kw * self.timestep_hours,

            "Battery_charge_kwh": total_battery_charge_kw * self.timestep_hours,

            # VPP power balance
            "VPP_net_power_kw": vpp_net_power_kw,
            "VPP_grid_import_kw": vpp_grid_import_kw,
            "VPP_grid_export_kw": vpp_grid_export_kw,
            
            # External demonstration resources
            "Hospital_load_kw": total_local_load_kw,
            "Wind_generation_kw": total_wind_generation_kw,
            
            # Wider demonstration/network power balance
            "Network_net_power_kw": network_net_power_kw,
            "Network_grid_import_kw": network_grid_import_kw,
            "Network_grid_export_kw": network_grid_export_kw,

            "VPP_mode": signals["mode"],
            "is_peak_period": signals["is_peak_period"],
            "Is_event_period": signals["is_event_period"],
            "Is_cheap_energy_period":signals["is_cheap_energy_period"],
            "PV_sharing_enabled": signals["pv_sharing_enabled"],

            "Requested_support_kw": signals["requested_support_kw"],
            "Available_VPP_discharge_kw":signals["total_available_discharge_kw"],
            
            "Shared_PV_available_kw": signals["total_shared_pv_available_kw"],
            "Shared_PV_transferred_kw": signals["total_shared_pv_transferred_kw"],
            
            "House_charge_targets": signals["house_charge_targets"],
            "House_shared_export_targets": signals["house_shared_export_targets"],
            
        }


        # Store summary result
        self.simulation_results_summary.append(timestep_result)


        # Store each individual house result
        for result in house_results:
            house_name = result["name"]
            self.house_results_log[house_name].append(result)


        # Advance simulation clock
        self.current_step += 1


        # Return the latest result
        return timestep_result


class House:
    def __init__(
        self,
        name,
        load_profile,
        generation_profile,
        battery=None,
        pv_array_kwp=0.0,
        pv_inverter_limit_kw=0.0,
        connection_type="single_phase",
        connection_phases=1,
        min_export_kw_per_phase=1.5,
        max_export_kw_per_phase=10.0
    ):

        self.name = name
        self.load_profile = load_profile
        self.generation_profile = generation_profile
        self.battery = battery

        # PV system
        self.pv_array_kwp = pv_array_kwp
        self.pv_inverter_limit_kw = pv_inverter_limit_kw

        # Grid connection
        self.connection_type = connection_type
        self.connection_phases = connection_phases

        self.min_export_kw_per_phase = min_export_kw_per_phase
        self.max_export_kw_per_phase = max_export_kw_per_phase

        # Aggregate connection limits used by the simplified model
        self.min_export_limit_kw = (
            min_export_kw_per_phase * connection_phases
        )

        self.max_export_limit_kw = (
            max_export_kw_per_phase * connection_phases
        )

        # Current dynamic export allowance.
        # Initially set to the maximum connection allowance.
        # This could later become a time-varying network input.
        
        # A04 - Normal dynamic export limit.
        # This value may later be updated by a DNSP/network signal.
        self.dynamic_export_limit_kw = self.max_export_limit_kw
        
        # Effective export limit actually enforced by the simulation.
        # A10 allows this to be temporarily increased during a
        # network-support event.
        self.current_export_limit_kw = self.dynamic_export_limit_kw
    
            
        
    def get_state(self, profile_index, timestep_hours):
        load_kw = self.load_profile[profile_index]
        generation_kw = self.generation_profile[profile_index]

        net_power_kw_before_battery = load_kw - generation_kw
        battery_soc_kwh = None
        battery_soc_percent = None
        max_discharge_kw = 0.0
        max_charge_kw = 0.0
        available_discharge_kw = 0.0
        available_charge_kw = 0.0
        export_headroom_kw = 0.0
        vpp_available_discharge_kw = 0.0
        local_pv_surplus_kw = max(
            -net_power_kw_before_battery,
            0.0
        )
        local_battery_charge_from_pv_kw = 0.0
        shared_pv_available_kw = local_pv_surplus_kw
        vpp_available_charge_kw = 0.0

        if self.battery is not None:

            battery_soc_kwh = self.battery.get_soc_kwh()
            battery_soc_percent = self.battery.get_soc_percent()

            max_discharge_kw = self.battery.max_discharge_kw
            max_charge_kw = self.battery.max_charge_kw

            available_energy_kwh = max(
                
                0.0,
                self.battery.soc_kwh - self.battery.min_soc_kwh
            )
            available_discharge_kw = min(
                self.battery.max_discharge_kw,
                available_energy_kwh * self.battery.efficiency / timestep_hours
            )
            
            # --------------------------------------------------
            # Connection-constrained VPP discharge capability
            # --------------------------------------------------
        
            local_import_kw = max(
                net_power_kw_before_battery,
                0.0
            )
        
            export_before_battery_kw = max(
                -net_power_kw_before_battery,
                0.0
            )
        
            export_headroom_kw = max(
                self.current_export_limit_kw
                - export_before_battery_kw,
                0.0
            )
        
            connection_permitted_discharge_kw = (
                local_import_kw
                + export_headroom_kw
            )
        
            vpp_available_discharge_kw = min(
                available_discharge_kw,
                connection_permitted_discharge_kw
            )

            # --------------------------------------------------
        
            
            available_space_kwh = max(
                0.0,
                self.battery.capacity_kwh - self.battery.soc_kwh
            )
            
            available_charge_kw = min(
                self.battery.max_charge_kw,
                available_space_kwh / (
                    timestep_hours * self.battery.efficiency
                )
            )
            
            
            local_battery_charge_from_pv_kw = min(
                local_pv_surplus_kw,
                available_charge_kw
            )
            
            shared_pv_available_kw = max(
                local_pv_surplus_kw - local_battery_charge_from_pv_kw,
                0.0
            )
            
            vpp_available_charge_kw = max(
                available_charge_kw - local_battery_charge_from_pv_kw,
                0.0
            )
                 

        return {
            "name": self.name,
            "load_kw": load_kw,
            "generation_kw": generation_kw,
            "net_power_kw_before_battery": net_power_kw_before_battery,

            "has_battery": self.battery is not None,
            
            "battery_soc_kwh": battery_soc_kwh,
            "battery_soc_percent": battery_soc_percent,
            
            "max_discharge_kw": max_discharge_kw,
            "max_charge_kw": max_charge_kw,

            "available_discharge_kw": available_discharge_kw,
            "available_charge_kw": available_charge_kw,

            "local_pv_surplus_kw": local_pv_surplus_kw,
            "local_battery_charge_from_pv_kw": local_battery_charge_from_pv_kw,
            "shared_pv_available_kw": shared_pv_available_kw,
            "vpp_available_charge_kw": vpp_available_charge_kw,
        
            "export_headroom_kw": export_headroom_kw,

            "vpp_available_discharge_kw": vpp_available_discharge_kw,

        
            "pv_array_kwp": self.pv_array_kwp,
            "pv_inverter_limit_kw": self.pv_inverter_limit_kw,
            
            "connection_type": self.connection_type,
            "connection_phases": self.connection_phases,
            
            "min_export_limit_kw": self.min_export_limit_kw,
            "max_export_limit_kw": self.max_export_limit_kw,

            "dynamic_export_limit_kw": self.dynamic_export_limit_kw,
            "current_export_limit_kw": self.current_export_limit_kw,
            
            }
            
    def update_export_limit(self, network_support_request=False):
        """
        Update the effective connection export limit.
    
        A10: During an explicit network-support event, the
        demonstrator assumes the normal dynamic export limit
        may be temporarily increased to the modelled maximum
        connection export capability.
        """
        if network_support_request:
            self.current_export_limit_kw = self.max_export_limit_kw
        else:
            self.current_export_limit_kw = self.dynamic_export_limit_kw                

    def get_export_headroom_kw(
            self,
            load_kw,
            generation_kw
    ):
        """
        Calculate how much additional battery discharge can occur
        without exceeding the current connection export limit.
    
        Positive net power = grid import.
        Negative net power = grid export.
        """
    
        net_before_battery_kw = (
            load_kw - generation_kw
        )
    
        export_before_battery_kw = max(
            -net_before_battery_kw,
            0.0
        )
    
        export_headroom_kw = max(
            self.current_export_limit_kw
            - export_before_battery_kw,
            0.0
        )
    
        return export_headroom_kw


        

    def step(
        self,
        t,
        timestep_hours,
        signals,
    ):
        load_kw = self.load_profile[t]
        generation_kw = self.generation_profile[t]
        # check if there is a surplus of generation
        net_power_kw_bb = load_kw - generation_kw
        
        # --------------------------------------------------
        # Connection-point state before battery action
        # --------------------------------------------------
        
        current_export_limit_kw = self.current_export_limit_kw

        # --------------------------------------------------
        # A09 - VPP-directed shared PV export allowance
        # --------------------------------------------------
        
        shared_pv_export_target_kw = signals[
            "house_shared_export_targets"
        ].get(
            self.name,
            0.0
        )

        # A09 - The VPP controller may override the normal
        # dynamic export limit when shared PV has been
        # allocated to other participating houses.
        effective_export_limit_kw = min(
            max(
                current_export_limit_kw,
                shared_pv_export_target_kw
            ),
            self.max_export_limit_kw
        )
                
        
        export_before_battery_kw = max(
            -net_power_kw_bb,
            0.0
        )

                
        export_headroom_kw = max(
            current_export_limit_kw
            - export_before_battery_kw,
            0.0
        )
        
        battery_discharge_reduction_kw = 0.0
        pv_curtailment_kw = 0.0
        export_limit_active = False

        
        # initialise battery power as 0 incase load meets demand
        
        battery_power_kw = 0.0
        
        # initialise battery_soc_kwh and percent incase no battery detected on house these values are local only and the true values exist within the objects themselves
        battery_soc_kwh = 0.0
        battery_soc_percent = 0.0
        
        
        # check to see if there is a battery apart of this household
        if self.battery is not None:
            # check what mode the VPP controller is instructing and execute actions
            mode = signals["mode"]
            is_peak_period = signals["is_peak_period"]
            is_grid_peak_limit_exceeded = signals["is_grid_peak_limit_exceeded"]
            is_event_period = signals["is_event_period"]
            is_cheap_energy_period = signals["is_cheap_energy_period"]

            
            # shared_pv_charge_target_kw = signals["house_charge_targets"].get(self.name,0.0)
            shared_pv_allocation_kw = signals["house_charge_targets"].get(self.name, 0.0)
            
            effective_net_power_kw = (net_power_kw_bb - shared_pv_allocation_kw)
            
            if mode == "self_consumption":

                if effective_net_power_kw < 0:
                    # charge from remaining surplus
                    battery_power_kw = self.battery.charge(-effective_net_power_kw, timestep_hours)

            
                elif effective_net_power_kw > 0:
                    # discharge for remaining deficit
                    battery_power_kw = self.battery.discharge(effective_net_power_kw, timestep_hours)
                
                else: 
                    battery_power_kw = 0.0
            
               
            elif mode == "battery_hold":
                battery_power_kw = 0.0
                
            elif mode == "peak_shaving":

                if effective_net_power_kw < 0:
                    battery_power_kw = self.battery.charge(
                        -effective_net_power_kw,
                        timestep_hours
                    )
            
                elif (
                    effective_net_power_kw > 0
                    and (
                        is_peak_period
                        or is_grid_peak_limit_exceeded
                    )
                ):
                    battery_power_kw = self.battery.discharge(
                        effective_net_power_kw,
                        timestep_hours
                    )
            
                else:
                    battery_power_kw = 0.0

                    
            elif mode == "export_support":
                battery_power_kw = self.battery.discharge(self.battery.max_discharge_kw, timestep_hours)
                # NOT USED CURRENTLY, ACTIVATE WHEN ECONOMIC MODEELING IS IMPLEMENTED
    
            elif mode == "local_load_support":

                requested_target_kw = signals[
                    "house_dispatch_targets"
                ].get(
                    self.name,
                    0.0
                )
            
                # Battery discharge can first offset any local import.
                local_import_kw = max(
                    effective_net_power_kw,
                    0.0
                )
            
                # Once local import is removed, any additional battery
                # discharge becomes grid export and must fit inside the
                # connection export allowance.
                maximum_connection_permitted_discharge_kw = (
                    local_import_kw
                    + export_headroom_kw
                )
            
                permitted_target_kw = min(
                    requested_target_kw,
                    maximum_connection_permitted_discharge_kw
                )
            
                battery_discharge_reduction_kw = max(
                    requested_target_kw
                    - permitted_target_kw,
                    0.0
                )
            
                battery_power_kw = self.battery.discharge(
                    permitted_target_kw,
                    timestep_hours
                )
                  
            elif mode == "arbitrage_charge":
                if is_cheap_energy_period:
                    battery_power_kw = self.battery.charge(self.battery.max_charge_kw, timestep_hours)

            
                
                
            else: 
                battery_power_kw = 0.0
            
            # update battery SOC values
            battery_soc_kwh = self.battery.get_soc_kwh()
            battery_soc_percent = self.battery.get_soc_percent()
                   
        # Update battery SOC values after all battery actions     
        battery_discharge_kw = max(battery_power_kw, 0.0)
        battery_charge_kw = -min(battery_power_kw, 0.0)

        # --------------------------------------------------
        # Potential connection-point power
        # --------------------------------------------------
        
        potential_net_power_kw = (
            load_kw
            - generation_kw
            - battery_power_kw
        )
        
        potential_export_kw = max(
            -potential_net_power_kw,
            0.0
        )
        
        # --------------------------------------------------
        # Enforce connection export limit
        # --------------------------------------------------
        
        if potential_export_kw > effective_export_limit_kw:
        
            export_limit_active = True
        
            excess_export_kw = (
                potential_export_kw
                - effective_export_limit_kw
            )
        
            # At this stage battery discharge has already been
            # constrained before SOC was updated.
            #
            # Therefore any remaining excess comes from PV and
            # must be curtailed.
        
            pv_curtailment_kw = min(
                excess_export_kw,
                generation_kw
            )
        
        else:
        
            export_limit_active = (
                battery_discharge_reduction_kw > 0
            )


        # Define actual power generation
        actual_generation_kw = (
            generation_kw
            - pv_curtailment_kw
        )
        # Now calculate the power the VPP sees cross the connection point
        
        final_net_power_kw = (
            load_kw
            - actual_generation_kw
            - battery_power_kw
        )
        
        # Battery metadata for logging/export
        if self.battery is not None:
            battery_manufacturer = self.battery.manufacturer
            battery_model = self.battery.model
            battery_module_count = self.battery.module_count
            battery_capacity_kwh = self.battery.capacity_kwh
            battery_max_charge_kw = self.battery.max_charge_kw
            battery_max_discharge_kw = self.battery.max_discharge_kw
        
        else:
            battery_manufacturer = None
            battery_model = None
            battery_module_count = 0
            battery_capacity_kwh = 0.0
            battery_max_charge_kw = 0.0
            battery_max_discharge_kw = 0.0


            
        return {
            "name": self.name,
            


            

            # --------------------------------------------------
            # Physical house configuration
            # --------------------------------------------------
            "pv_array_kwp": self.pv_array_kwp,
            "pv_inverter_limit_kw": self.pv_inverter_limit_kw,
    
            "connection_type": self.connection_type,
            "connection_phases": self.connection_phases,
            "min_export_limit_kw": self.min_export_limit_kw,
            "max_export_limit_kw": self.max_export_limit_kw,
            "dynamic_export_limit_kw": self.dynamic_export_limit_kw,
            
            # --------------------------------------------------
            # Battery configuration
            # --------------------------------------------------
            "battery_manufacturer": battery_manufacturer,
            "battery_model": battery_model,
            "battery_module_count": battery_module_count,
            "battery_capacity_kwh": battery_capacity_kwh,
            "battery_max_charge_kw": battery_max_charge_kw,
            "battery_max_discharge_kw": battery_max_discharge_kw,
        
            # --------------------------------------------------
            # Timestep results
            # --------------------------------------------------
            "load_kw": load_kw,
    
            # PV generation
            "available_generation_kw": generation_kw,  # what the PV could have acutally produced
            "pv_curtailment_kw": pv_curtailment_kw,    # what could not be used or exported
            "generation_kw": actual_generation_kw,    # what was actually used by the sim/grid/system

            # Connection-point results
            "potential_net_power_kw": potential_net_power_kw,
            
            "current_export_limit_kw": current_export_limit_kw,
            # A09
            "shared_pv_export_target_kw": shared_pv_export_target_kw,
            "effective_export_limit_kw": effective_export_limit_kw,

            
            "export_limit_active": export_limit_active,
            "battery_discharge_reduction_kw":battery_discharge_reduction_kw,

            
            "net_power_kw_before_battery": net_power_kw_bb,
        
            "battery_power_kw": battery_power_kw,
            "battery_soc_kwh": battery_soc_kwh,
            "battery_soc_percent": battery_soc_percent,
        
            "net_power_kw": final_net_power_kw,
        
            "battery_charge_kw": battery_charge_kw,
            "battery_discharge_kw": battery_discharge_kw,
            }


        
#how do I improve load and generation simulation? need realistic load and genereation profiles for each house depending on their availiable resources. could also reseach files to obtain accurrate hourly load profiles (clean data in Excel)

# Create the Battery Class/Object, This may later be moved inside of the house class once tested for behaviour

class Battery:
    def __init__(
        self,
        capacity_kwh,
        initial_soc_kwh,
        max_charge_kw,
        max_discharge_kw,
        efficiency=0.95,
        min_soc_kwh=0.0,
        manufacturer=None,
        model=None,
        module_count=None
    ):
        
        self.capacity_kwh = capacity_kwh
        self.soc_kwh = initial_soc_kwh
    
        self.max_charge_kw = max_charge_kw
        self.max_discharge_kw = max_discharge_kw
    
        self.efficiency = efficiency
        self.min_soc_kwh = min_soc_kwh
    
        self.manufacturer = manufacturer
        self.model = model
        self.module_count = module_count
        
    def charge(self, requested_power_kw, timestep_hours):
        # check requested charge power is positive
        if requested_power_kw <= 0:
            return 0.0
        # limit requested power
        charge_power_kw = min(requested_power_kw, self.max_charge_kw)
        # convert to stored energy
        requested_energy_kwh = charge_power_kw *timestep_hours*self.efficiency
        # check availiable battery space
        available_space_kwh = self.capacity_kwh - self.soc_kwh
        # limit actual stored energy to within max capacity
        actual_energy_kwh = min(requested_energy_kwh, available_space_kwh)
        # update the SOC
        self.soc_kwh += actual_energy_kwh
        # calculate the charging power
        actual_charge_power_kw = actual_energy_kwh / (timestep_hours*self.efficiency)
        #return value of power stored by the battery to the simulation
        battery_power_kw = actual_charge_power_kw*-1
        return battery_power_kw
        
    def discharge(self, requested_power_kw, timestep_hours):
         # check requested discharge power is positive
        if requested_power_kw <= 0:
            return 0.0
        # limit requested discharge power
        discharge_power_kw = min(requested_power_kw, self.max_discharge_kw)
        # convert requested power to battery energy removed
        requested_energy_kwh = discharge_power_kw * timestep_hours / self.efficiency
        # check current SOC to determine if battery has available capacity
        available_energy_kwh = max(0.0, self.soc_kwh - self.min_soc_kwh)
        # limit discharged power to what is actually available
        actual_energy_kwh = min(available_energy_kwh, requested_energy_kwh) 
        # update the SOC
        self.soc_kwh -= actual_energy_kwh
        # calculate the discharge power
        actual_discharge_power_kw = (actual_energy_kwh*self.efficiency)/timestep_hours
        # return value of power discharged from the battery
        battery_power_kw = actual_discharge_power_kw
        return battery_power_kw
        
    def get_soc_percent(self):
        return (self.soc_kwh/self.capacity_kwh)*100
    def get_soc_kwh(self):
        return self.soc_kwh

# Define the VPP controller class
class VPPController:
    def __init__(self, 
                 default_mode="self_consumption", 
                 peak_hours=None, 
                 event_hours=None, 
                 cheap_energy_hours=None,
                 grid_import_limit_kw=None
                ):
        
        self.default_mode = default_mode
        self.peak_hours = peak_hours or []
        self.event_hours = event_hours or []
        self.cheap_energy_hours = cheap_energy_hours or []
        self.grid_import_limit_kw = grid_import_limit_kw

    def get_signals(
        self, 
        t, 
        external_inputs=None, 
        house_states=None,
        hospital_state=None,
        wind_state=None
    ):

        if external_inputs is None:
            external_inputs = {}

        if house_states is None:
            house_states = []


         # Calculate grid demand before any battery action
        total_house_net_before_battery_kw = sum(
            state["net_power_kw_before_battery"]
            for state in house_states
        )
        total_shared_pv_available_kw = sum(
            state["shared_pv_available_kw"]
            for state in house_states
        )

        total_vpp_available_charge_kw = sum(
            state["vpp_available_charge_kw"]
            for state in house_states
        )
        
        # ---------------------------------------------------------
        # VPP electrical state
        # ---------------------------------------------------------
        # The VPP consists only of the participating houses
        # and their DERs.
        #
        # Hospital and wind/DFIG are external demonstration
        # resources and must not be included in the VPP's
        # internal power balance.
        
        vpp_net_power_before_battery_kw = (
            total_house_net_before_battery_kw
        )
        
        vpp_grid_import_before_battery_kw = max(
            vpp_net_power_before_battery_kw,
            0.0
        )

         # Existing external input logic
        network_support_request = external_inputs.get(
            "network_support_request",
            False
        )

        requested_support_kw = external_inputs.get(
            "requested_support_kw",
            0.0
        )

        house_dispatch_targets = {}
        
        house_charge_targets = {}

        
        # A09 - Records how much shared PV is attributed
        # to each surplus-producing VPP house.
        house_shared_export_targets = {}

         # Calculate current VPP battery availability
        total_available_discharge_kw = sum(
            state["vpp_available_discharge_kw"]
            for state in house_states
        )
        
        peak_shaving_enabled = external_inputs.get("peak_shaving_enabled", False)

        pv_sharing_enabled = external_inputs.get(
            "pv_sharing_enabled",
            False
        )

        
        manual_mode = external_inputs.get("manual_mode")

        if manual_mode is not None:
            active_mode = manual_mode
        
        elif network_support_request:
            active_mode = "local_load_support"

            available_houses = [
                state for state in house_states
                if state["has_battery"] 
                and state["vpp_available_discharge_kw"] > 0
            ]

            remaining_support_kw = requested_support_kw
    
            while available_houses and remaining_support_kw > 0:
    
                equal_share_kw = (remaining_support_kw/len(available_houses))
    
                still_available_houses = []
        
                for state in available_houses:
        
                    house_name = state["name"]
                    available_kw = state["vpp_available_discharge_kw"]
        
                   

                    already_dispatched_kw = (house_dispatch_targets.get(
                        house_name,
                        0.0
                        )
                    )

                    remaining_house_capacity_kw = (available_kw - already_dispatched_kw)

                    dispatch_kw = min(
                        equal_share_kw,
                        remaining_house_capacity_kw
                    )
        
                    house_dispatch_targets[house_name] = (already_dispatched_kw + dispatch_kw)
        
                    remaining_support_kw -= dispatch_kw
        
                    remaining_house_capacity_kw -= dispatch_kw
        
                    if remaining_house_capacity_kw > 0:
                        still_available_houses.append(state)
        
                available_houses = still_available_houses
        elif t in self.cheap_energy_hours:
            active_mode = "arbitrage_charge"
        
        elif peak_shaving_enabled and t in self.peak_hours:
            active_mode = "peak_shaving"
        
        elif (
            self.grid_import_limit_kw is not None
            and vpp_grid_import_before_battery_kw > self.grid_import_limit_kw
        ):
            active_mode = "peak_shaving"

            
        else:
            active_mode = self.default_mode

        # ---------------------------------------------------------
        # Secondary VPP action: redistribute genuine excess PV
        # ---------------------------------------------------------
        
        shared_pv_charging_allowed = (
            pv_sharing_enabled
            and active_mode in ["self_consumption", "peak_shaving"]
            and not (peak_shaving_enabled and t in self.peak_hours)
        )

        
        if (
            shared_pv_charging_allowed
            and total_shared_pv_available_kw > 0
            and total_vpp_available_charge_kw > 0
        ):
        
            available_houses = [
                state for state in house_states
                if state["has_battery"]
                and state["vpp_available_charge_kw"] > 0
            ]
        
            remaining_shared_pv_kw = total_shared_pv_available_kw
        
            while available_houses and remaining_shared_pv_kw > 0:
        
                equal_share_kw = (
                    remaining_shared_pv_kw / len(available_houses)
                )
        
                still_available_houses = []
        
                for state in available_houses:
        
                    house_name = state["name"]
                    available_kw = state["vpp_available_charge_kw"]
        
                    already_allocated_kw = house_charge_targets.get(
                        house_name,
                        0.0
                    )
        
                    remaining_house_capacity_kw = (
                        available_kw - already_allocated_kw
                    )
        
                    charge_kw = min(
                        equal_share_kw,
                        remaining_house_capacity_kw
                    )
        
                    house_charge_targets[house_name] = (
                        already_allocated_kw + charge_kw
                    )
        
                    remaining_shared_pv_kw -= charge_kw
                    remaining_house_capacity_kw -= charge_kw
        
                    if remaining_house_capacity_kw > 0:
                        still_available_houses.append(state)
        
                available_houses = still_available_houses

        # ---------------------------------------------------------
        # A09 - Actual VPP-directed shared PV transfer
        # ---------------------------------------------------------
        # Distinguish between PV that was available for sharing
        # and PV that was actually allocated to VPP batteries.

        total_shared_pv_transferred_kw = sum(
            house_charge_targets.values()
        )
        # ---------------------------------------------------------
        # A09 - Attribute transferred PV back to source houses
        # ---------------------------------------------------------
        
        remaining_shared_export_kw = total_shared_pv_transferred_kw
        
        for state in house_states:
        
            if remaining_shared_export_kw <= 0:
                break
        
            source_available_kw = state["shared_pv_available_kw"]
        
            if source_available_kw <= 0:
                continue
        
            source_export_kw = min(
                source_available_kw,
                remaining_shared_export_kw
            )
        
            house_shared_export_targets[state["name"]] = source_export_kw
        
            remaining_shared_export_kw -= source_export_kw


        return{
            "mode":active_mode,
            "is_peak_period": t in self.peak_hours,
            "is_grid_peak_limit_exceeded": (self.grid_import_limit_kw is not None and vpp_grid_import_before_battery_kw > self.grid_import_limit_kw),
            "peak_shaving_enabled": peak_shaving_enabled,
            "pv_sharing_enabled": pv_sharing_enabled,

            "is_event_period": network_support_request or t in self.event_hours,
            "is_cheap_energy_period":t in self.cheap_energy_hours,

            "requested_support_kw": requested_support_kw,
            "house_dispatch_targets": house_dispatch_targets,
            "vpp_grid_import_before_battery_kw": vpp_grid_import_before_battery_kw,
            # Current VPP capability
            
            "total_available_discharge_kw": total_available_discharge_kw,
            
            "total_shared_pv_transferred_kw": total_shared_pv_transferred_kw,
            "total_shared_pv_available_kw": total_shared_pv_available_kw,
            "total_vpp_available_charge_kw": total_vpp_available_charge_kw,
            
            "house_charge_targets": house_charge_targets,
            "house_shared_export_targets": house_shared_export_targets,
        }
        


#define a local load that the VPP can supply this is currently a basic place holder that will be updated later to match realistic demand values and have its own demand profile (currently static) 

class LocalLoad:
    def __init__(self, name, demand_kw):
        self.name = name
        self.demand_kw = demand_kw

    def step(self, t):
        return {
            "name": self.name,
            "load_kw": self.demand_kw[t]
        }


class WindTurbine:
    def __init__(self, name, generation_profile):
        self.name = name
        self.generation_profile = generation_profile

    def step(self, t):
        return {
            "name": self.name,
            "generation_kw": self.generation_profile[t]
        }

#wind turbine could have three different generation settings based on the wind speed (no wind/lock out (too much wind), moderate wind, ideal wind)
#middle setting could have variance to show slight hour to hour changes perhaps?
#need to confirm how this works in realworld -> further research required





In [14]:
def get_hardware_states(simulation):
    """
    Convert the latest simulated battery results
    into states for the physical LEDs.
    """

    states = {}

    for house in simulation.houses:

        # House 3 has no battery
        if house.battery is None:
            states[house.name] = "no_battery"
            continue

        # Get this house's result from the timestep
        # that has just been simulated
        latest = simulation.house_results_log[house.name][-1]

        soc_kwh = latest["battery_soc_kwh"]
        charge_kw = latest["battery_charge_kw"]
        discharge_kw = latest["battery_discharge_kw"]

        max_soc_kwh = house.battery.capacity_kwh
        min_soc_kwh = house.battery.min_soc_kwh

        # Small tolerance avoids floating-point comparison problems
        tolerance = 1e-6

        if soc_kwh >= max_soc_kwh - tolerance:
            states[house.name] = "full"

        elif soc_kwh <= min_soc_kwh + tolerance:
            states[house.name] = "minimum"

        elif charge_kw > tolerance:
            states[house.name] = "charging"

        elif discharge_kw > tolerance:
            states[house.name] = "discharging"

        else:
            states[house.name] = "idle"

    return states

Simulation Loop

In [15]:
# ============================================================
# REALISTIC PROFILE DATA
# ============================================================

# Simulation resolution
timestep_hours = 0.5
num_steps = len(summer_data)
simulation_hours = num_steps * timestep_hours



# ============================================================
# EXTERNAL DEMONSTRATION PLACEHOLDER PROFILES
# ============================================================

# Hospital placeholder - constant demand for now
hospital_load_profile = np.full(
    num_steps,
    6.0
)

# Wind placeholder - zero generation for now
wind_generation_profile = np.zeros(
    num_steps
)



# ----------------------------
# Household load profiles
# ----------------------------

house_1_load = summer_data["H1_HI_B_kW"].to_numpy()
house_2_load = summer_data["H2_MED_B_kW"].to_numpy()
house_3_load = summer_data["H3_LOW_A_Kw"].to_numpy()
house_4_load = summer_data["H4_Hi_A_kW"].to_numpy()
house_5_load = summer_data["H5_Med_A_kW"].to_numpy()


# ----------------------------
# PV generation profiles
# ----------------------------

house_1_pv = summer_data["PV_13.3kW_kW"].to_numpy()
house_2_pv = summer_data["PV_6.6kW_kW"].to_numpy()
house_3_pv = summer_data["PV_13.3kW_kW"].to_numpy()
house_4_pv = summer_data["PV_20kW_kW"].to_numpy()

# House 5 has no PV
house_5_pv = np.zeros(num_steps)


load_timestamps = pd.to_datetime(
    summer_data["LOAD_SOURCE_TIMESTEP"],
    dayfirst=True
)


pv_timestamps = pd.to_datetime(
    summer_data["PV_SOURCE_TIMESTAMP"],
    dayfirst=True
)

print("\n=== TIMESTAMP CHECK ===")
print("Load start:", load_timestamps.iloc[0])
print("Load end:  ", load_timestamps.iloc[-1])
print("PV start:  ", pv_timestamps.iloc[0])
print("PV end:    ", pv_timestamps.iloc[-1])

#need to add Wind Turbine and industrial load profiles here potentially? UPDATE WIND GEN AND LOCAL LOAD CLASS WHEN THIS IS DONE

# Verify all profiles are the same length
profiles = [
    house_1_load,
    house_2_load,
    house_3_load,
    house_4_load,
    house_5_load,
    house_1_pv,
    house_2_pv,
    house_3_pv,
    house_4_pv,
    house_5_pv,
    hospital_load_profile,
    wind_generation_profile,
]
profile_lengths = set([len(profile) for profile in profiles])


if len(profile_lengths) > 1 :
    raise ValueError("Profiles are not the same length")
else :
    print("All Load and Generation Profiles are the same length")

print("Number of Simulation steps=", num_steps)
print("Number of Hours Simulated=", simulation_hours) 

# --------------------------------------------------
# SigenStor battery simulation assumptions
# --------------------------------------------------
#
# Battery model based on Sigen Battery 9.0 module:
#   Usable energy per module: 8.76 kWh
#   Continuous charge/discharge power per module: 4.6 kW
#
# Simulation assumptions:
#   - Module energy and power capability are cumulative.
#   - BESS inverter charge/discharge power is capped at 10 kW.
#   - Manufacturer usable capacity represents the simulated
#     0–100% SOC operating range.
#   - Any future VPP SOC reserve will be implemented by the
#     controller, not by reducing physical battery capacity.

SIGEN_MODULE_USABLE_KWH = 8.76
SIGEN_MODULE_POWER_KW = 4.6

BESS_INVERTER_LIMIT_KW = 10.0


def create_sigenstor_spec(module_count, initial_soc_percent=50.0):

    usable_capacity_kwh = (
        module_count * SIGEN_MODULE_USABLE_KWH
    )

    raw_stack_power_kw = (
        module_count * SIGEN_MODULE_POWER_KW
    )

    max_bess_power_kw = min(
        raw_stack_power_kw,
        BESS_INVERTER_LIMIT_KW
    )

    initial_soc_kwh = (
        usable_capacity_kwh
        * initial_soc_percent / 100.0
    )

    return {
        "capacity_kwh": usable_capacity_kwh,
        "initial_soc_kwh": initial_soc_kwh,

        "max_charge_kw": max_bess_power_kw,
        "max_discharge_kw": max_bess_power_kw,

        "efficiency": 0.95,

        # 0% represents the bottom of the manufacturer's
        # stated usable energy range.
        "min_soc_kwh": 0.0,
        
        "manufacturer": "Sigenergy",
        "model": "SigenStor BAT 9.0",
        "module_count": module_count,
    }



=== TIMESTAMP CHECK ===
Load start: 2013-02-04 00:00:00
Load end:   2013-02-10 23:30:00
PV start:   2013-01-07 00:00:00
PV end:     2013-01-13 23:30:00
All Load and Generation Profiles are the same length
Number of Simulation steps= 336
Number of Hours Simulated= 168.0


In [16]:

# Self-consumption is the normal operating mode.
# Tariff peak shaving can be enabled through external_inputs.

controller = VPPController(
    default_mode="self_consumption",
    peak_hours=[
        16.0, 16.5,
        17.0, 17.5,
        18.0, 18.5,
        19.0, 19.5,
        20.0, 20.5,
    ],
    event_hours=[],
    cheap_energy_hours=[]
)
scenario_name="external_network_support_test"


# Initialise Batteries

# --------------------------------------------------
# Initialise SigenStor battery systems
# --------------------------------------------------

# H1 - High-DER prosumer
# 3 x Sigen Battery 9.0 modules
house1_battery = Battery(
    **create_sigenstor_spec(
        module_count=3,
        initial_soc_percent=50.0
    )
)

# H2 - Typical prosumer
# 2 x Sigen Battery 9.0 modules
house2_battery = Battery(
    **create_sigenstor_spec(
        module_count=2,
        initial_soc_percent=50.0
    )
)

# H3 - PV-only household
# No battery

# H4 - High-demand / future three-phase household
# 3 x Sigen Battery 9.0 modules
house4_battery = Battery(
    **create_sigenstor_spec(
        module_count=3,
        initial_soc_percent=50.0
    )
)

# H5 - Battery-heavy / no-PV household
# 5 x Sigen Battery 9.0 modules
house5_battery = Battery(
    **create_sigenstor_spec(
        module_count=5,
        initial_soc_percent=50.0
    )
)



# Initialise Houses
houses = [
   
    # H1 - High-DER prosumer
    House(
        name="House 1",
        load_profile=house_1_load,
        generation_profile=house_1_pv,
        battery=house1_battery,
        pv_array_kwp=13.3,
        pv_inverter_limit_kw=10.64,
        connection_type="single_phase",
        connection_phases=1
    ),

    # H2 - Typical prosumer
    House(
        name="House 2",
        load_profile=house_2_load,
        generation_profile=house_2_pv,
        battery=house2_battery,
        pv_array_kwp=6.6,
        pv_inverter_limit_kw=5.28,
        connection_type="single_phase",
        connection_phases=1
    ),

    # H3 - PV-rich household, no battery
    House(
        name="House 3",
        load_profile=house_3_load,
        generation_profile=house_3_pv,
        battery=None,
        pv_array_kwp=13.3,
        pv_inverter_limit_kw=10.64,
        connection_type="single_phase",
        connection_phases=1
    ),

    # H4 - High-demand three-phase household
    House(
        name="House 4",
        load_profile=house_4_load,
        generation_profile=house_4_pv,
        battery=house4_battery,
        pv_array_kwp=20.0,
        pv_inverter_limit_kw=16.0,
        connection_type="three_phase",
        connection_phases=3
    ),

    # H5 - Battery-only household
    House(
        name="House 5",
        load_profile=house_5_load,
        generation_profile=house_5_pv,
        battery=house5_battery,
        pv_array_kwp=0.0,
        pv_inverter_limit_kw=0.0,
        connection_type="single_phase",
        connection_phases=1
    ),
]
# TEMPORARY CONNECTION-POINT TEST
# houses[0].current_export_limit_kw = 1.0

# houses[2].dynamic_export_limit_kw = 1.0
# houses[2].current_export_limit_kw = 1.0

# Initialise hospital and wind turbine

hospital = LocalLoad(
    "Hospital",
    hospital_load_profile
)

wind = WindTurbine(
    "DFIG Wind Placeholder",
    wind_generation_profile
)

# Create simulation
simulation = VPPSimulation(
    houses=houses,
    hospital=hospital,
    wind=wind,
    controller=controller,
    timestep_hours=timestep_hours,
    num_steps=num_steps
)

In [17]:
# --------------------------------------------------
# Continuous hardware demonstration
# --------------------------------------------------


# --------------------------------------------------
# Toggle PV sharing
# --------------------------------------------------







# --------------------------------------------------
# Simulated network support event
# --------------------------------------------------

event_day = 1
event_hour = 12
event_duration_hours = 2

event_start_step = int(
    (((event_day - 1) * 24) + event_hour)
    / timestep_hours
)

event_duration_steps = int(
    event_duration_hours / timestep_hours
)

event_end_step = (
    event_start_step
    + event_duration_steps
)

# --------------------------------------------------
# Simulated tariff peak-shaving toggle
# --------------------------------------------------

peak_toggle_day = 2
peak_toggle_hour = 10
peak_toggle_duration_hours = 12

peak_toggle_start_step = int(
    (((peak_toggle_day - 1) * 24) + peak_toggle_hour)
    / timestep_hours
)

peak_toggle_duration_steps = int(
    peak_toggle_duration_hours / timestep_hours
)

peak_toggle_end_step = (
    peak_toggle_start_step
    + peak_toggle_duration_steps
)

hardware = HardwareInterface()

try:
    print("Continuous VPP hardware demonstration started.")
    print("Use Jupyter interrupt/stop to finish the run.\n")

    while True:

        external_inputs = {}

        # Simulated peak-shaving strategy button
        peak_shaving_enabled = (
            peak_toggle_start_step
            <= simulation.current_step
            < peak_toggle_end_step
        )
        
        external_inputs["peak_shaving_enabled"] = peak_shaving_enabled
        external_inputs["pv_sharing_enabled"] = False

        # Simulated network-support event
        if (
            event_start_step
            <= simulation.current_step
            < event_end_step
        ):
            external_inputs["network_support_request"] = True
            external_inputs["requested_support_kw"] = 28.0
        # Run one logical simulation timestep
        result = simulation.step(external_inputs)

        # Convert latest simulation results
        # into LED states
        hardware_states = get_hardware_states(simulation)

        # Print useful live information
        print(
            f"\nStep {result['Simulation_step']} | "
            f"Day {result['Day_number']} | "
            f"Hour {result['Hour_of_day']:.1f} | "
            f"Mode: {result['VPP_mode']} | "
            f"Peak strategy: {'ON' if peak_shaving_enabled else 'OFF'}"
        )

        for house in simulation.houses:

            latest = simulation.house_results_log[house.name][-1]
            state = hardware_states[house.name]

            print(
                f"{house.name}: "
                f"{state:12s} | "
                f"SOC = {latest['battery_soc_kwh']:.2f} kWh | "
                f"Charge = {latest['battery_charge_kw']:.2f} kW | "
                f"Discharge = {latest['battery_discharge_kw']:.2f} kW"
            )
        print(
            "Shared PV available: "
            f"{result['Shared_PV_available_kw']:.2f} kW | "
            f"Charge targets: {result['House_charge_targets']}"
        )

        # Extra output during network-support event
        if result["VPP_mode"] == "local_load_support":

            print(
                "NETWORK SUPPORT | "
                f"Requested = {result['Requested_support_kw']:.2f} kW | "
                f"Available = {result['Available_VPP_discharge_kw']:.2f} kW | "
                f"Actual = {result['Battery_discharge_kw']:.2f} kW"
            )

        # Display this timestep on physical LEDs
        # for approximately 3 real seconds.
        hardware.display_states(
            hardware_states,
            duration=1.0
        )


except KeyboardInterrupt:

    print("\nSimulation interrupted by user.")


finally:

    hardware.cleanup()

    print("GPIO outputs cleaned up.")


# --------------------------------------------------
# Export everything accumulated during the run
# --------------------------------------------------

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

filename = f"SimulationResults_{scenario_name}_{timestamp}.xlsx"
filepath = f"results/{filename}"

summary_df = pd.DataFrame(
    simulation.simulation_results_summary
).round(2)

with pd.ExcelWriter(filepath, engine="openpyxl") as writer:

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for house_name, house_log in simulation.house_results_log.items():

        house_df = pd.DataFrame(house_log).round(2)

        sheet_name = house_name.replace(" ", "_")

        house_result_export = house_df.drop(
            columns=["name"]
        )

        house_result_export.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

print(
    simulation.current_step,
    "simulation steps completed"
)

print(
    f"Results exported to results/{filename}"
)

/home/jpobar/egh490_vpp_project/venv/lib/python3.12/site-packages/Jetson/GPIO/gpio_cdev.py:373: UserWarning: Could not open /dev/mem for pinmux check. If you want pinmux checks, make sure your user has permissions to read /dev/mem and that it exists. Error: [Errno 13] Permission denied: '/dev/mem'
  warnings.warn('Could not open /dev/mem for pinmux check. If you want pinmux checks, make sure your user has permissions to read /dev/mem and that it exists. Error: ' + str(e))


Continuous VPP hardware demonstration started.
Use Jupyter interrupt/stop to finish the run.


Step 0 | Day 1 | Hour 0.0 | Mode: self_consumption | Peak strategy: OFF
House 1: discharging  | SOC = 12.77 kWh | Charge = -0.00 kW | Discharge = 0.71 kW
House 2: discharging  | SOC = 8.68 kWh | Charge = -0.00 kW | Discharge = 0.15 kW
House 3: no_battery   | SOC = 0.00 kWh | Charge = -0.00 kW | Discharge = 0.00 kW
House 4: discharging  | SOC = 13.03 kWh | Charge = -0.00 kW | Discharge = 0.21 kW
House 5: discharging  | SOC = 21.71 kWh | Charge = -0.00 kW | Discharge = 0.35 kW
Shared PV available: 0.00 kW | Charge targets: {}

Step 1 | Day 1 | Hour 0.5 | Mode: self_consumption | Peak strategy: OFF
House 1: discharging  | SOC = 12.40 kWh | Charge = -0.00 kW | Discharge = 0.69 kW
House 2: discharging  | SOC = 8.60 kWh | Charge = -0.00 kW | Discharge = 0.15 kW
House 3: no_battery   | SOC = 0.00 kWh | Charge = -0.00 kW | Discharge = 0.00 kW
House 4: discharging  | SOC = 12.91 kWh | Charge = -0.00 k

In [12]:

# --------------------------------------------------
# REAL PROFILE DEBUG - Step 12 (06:00)
# --------------------------------------------------

debug_step = 12

print("\n=== STEP 12 / 06:00 HOUSE DEBUG ===")

for house in simulation.houses:

    result = simulation.house_results_log[
        house.name
    ][debug_step]

    print(
        f"\n{house.name}"
        f"\n  Load:                 {result['load_kw']:.3f} kW"
        f"\n  Available PV:         {result['available_generation_kw']:.3f} kW"
        f"\n  Net before battery:   {result['net_power_kw_before_battery']:.3f} kW"
        f"\n  Battery charge:       {result['battery_charge_kw']:.3f} kW"
        f"\n  Battery discharge:    {result['battery_discharge_kw']:.3f} kW"
        f"\n  Final net power:      {result['net_power_kw']:.3f} kW"
    )

print(
    "\nShared PV targets:",
    simulation.simulation_results_summary[
        debug_step
    ]["House_charge_targets"]
)








# # ---------------------------------------------------------
# # DEBUG VIEW - House 1 connection/export behaviour
# # ---------------------------------------------------------

# house_1_debug = pd.DataFrame(
#     simulation.house_results_log["House 1"]
# )

# debug_columns = [
#     "available_generation_kw",
#     "generation_kw",
#     "battery_discharge_kw",
#     "potential_net_power_kw",
#     "current_export_limit_kw",
#     "max_export_limit_kw",
#     "pv_curtailment_kw",
#     "net_power_kw",
#     "load_kw",
# ]

# print(
#     house_1_debug.loc[11:14, debug_columns].to_string()
# )

# DEBUG VIEW - Sim summary

# simulation_debug = pd.DataFrame(
#     simulation.simulation_results_summary
# )

# debug_columns = [
#     "Shared_PV_available_kw",
#     "Shared_PV_transferred_kw",
#     "House_shared_export_targets",
#     "House_charge_targets",
#     "VPP_net_power_kw",
#     "VPP_grid_import_kw",
#     "VPP_grid_export_kw",
# ]

# print(
#     simulation_debug.loc[11:14, debug_columns].to_string()
# )



=== STEP 12 / 06:00 HOUSE DEBUG ===

House 1
  Load:                 0.590 kW
  Available PV:         0.336 kW
  Net before battery:   0.254 kW
  Battery charge:       -0.000 kW
  Battery discharge:    0.193 kW
  Final net power:      0.061 kW

House 2
  Load:                 0.138 kW
  Available PV:         0.167 kW
  Net before battery:   -0.029 kW
  Battery charge:       0.089 kW
  Battery discharge:    0.000 kW
  Final net power:      0.061 kW

House 3
  Load:                 0.093 kW
  Available PV:         0.336 kW
  Net before battery:   -0.243 kW
  Battery charge:       -0.000 kW
  Battery discharge:    0.000 kW
  Final net power:      -0.243 kW

House 4
  Load:                 0.186 kW
  Available PV:         0.505 kW
  Net before battery:   -0.319 kW
  Battery charge:       0.380 kW
  Battery discharge:    0.000 kW
  Final net power:      0.061 kW

House 5
  Load:                 0.319 kW
  Available PV:         0.000 kW
  Net before battery:   0.319 kW
  Battery charge:    

In [8]:
house_3_debug = pd.DataFrame(
    simulation.house_results_log["House 3"]
)

debug_columns = [
    "load_kw",
    "available_generation_kw",
    "generation_kw",
    "potential_net_power_kw",
    "dynamic_export_limit_kw",
    "current_export_limit_kw",
    "shared_pv_export_target_kw",
    "effective_export_limit_kw",
    "max_export_limit_kw",
    "pv_curtailment_kw",
    "net_power_kw",
]

print(
    house_3_debug.loc[11:14, debug_columns].to_string()
)

    load_kw  available_generation_kw  generation_kw  potential_net_power_kw  dynamic_export_limit_kw  current_export_limit_kw  shared_pv_export_target_kw  effective_export_limit_kw  max_export_limit_kw  pv_curtailment_kw  net_power_kw
11      0.7                      5.2            5.2                    -4.5                      1.0                      1.0                         4.5                        4.5                 10.0                0.0          -4.5
12      0.8                      5.0            5.0                    -4.2                      1.0                     10.0                         0.0                       10.0                 10.0                0.0          -4.2
13      0.8                      4.5            4.5                    -3.7                      1.0                     10.0                         0.0                       10.0                 10.0                0.0          -3.7
14      0.9                      3.6            3.6         

In [18]:
for house in simulation.houses:
    latest = simulation.house_results_log[house.name][-1]

    print(
        f"{house.name}: "
        f"PV={latest['generation_kw']:.2f} kW, "
        f"Load={latest['load_kw']:.2f} kW, "
        f"Charge={latest['battery_charge_kw']:.2f} kW, "
        f"Discharge={latest['battery_discharge_kw']:.2f} kW, "
        f"SOC={latest['battery_soc_kwh']:.2f} kWh"
    )

print(
    "VPP mode:",
    simulation.simulation_results_summary[-1]["VPP_mode"]
)





House 1: PV=0.00 kW, Load=2.40 kW, Charge=-0.00 kW, Discharge=2.40 kW, SOC=21.33 kWh
House 2: PV=0.00 kW, Load=2.40 kW, Charge=-0.00 kW, Discharge=2.40 kW, SOC=9.78 kWh
House 3: PV=0.00 kW, Load=2.20 kW, Charge=-0.00 kW, Discharge=0.00 kW, SOC=0.00 kWh
House 4: PV=0.00 kW, Load=3.00 kW, Charge=-0.00 kW, Discharge=3.00 kW, SOC=18.07 kWh
House 5: PV=0.00 kW, Load=2.80 kW, Charge=-0.00 kW, Discharge=2.80 kW, SOC=16.08 kWh
VPP mode: self_consumption


In [17]:
# # run the sim 10 steps to 10am
# for _ in range(10):
#     simulation.step()
# #check the batteries for relevant states to display    
# hardware_states = get_hardware_states(simulation)
# print(hardware_states)

# #run the hardware interface to display the states physically on the LEDs
# hardware = HardwareInterface()

# try:
#     hardware.display_states(
#         hardware_states,
#         duration=3.0
#     )
# finally:
#     hardware.cleanup()






# # battery class testing
# battery_test = Battery(
#     capacity_kwh=10,
#     initial_soc_kwh=5.0,
#     max_charge_kw=2.0,
#     max_discharge_kw=2.0,
#     efficiency=0.95,
#     min_soc_kwh=1.0
# )



# timestep_hours = 1.0

# print(round(battery_test.get_soc_percent(),2),"%")
# charge_test = battery_test.charge(2.0, timestep_hours)
# print(round(charge_test,2))
# print(round(battery_test.get_soc_percent(),2),"%")


# discharge_test=battery_test.discharge(4.0, timestep_hours)
# print(round(discharge_test,2))
# print(round(battery_test.get_soc_percent(),2),"%")






# old test codes

# test_days = 5
# steps_to_run = int((24 / timestep_hours) * test_days)

# for _ in range(steps_to_run):
#     simulation.step()
    


# profile_index = simulation.current_step % simulation.num_steps

# for house in houses:
#     state = house.get_state(
#         profile_index,
#         timestep_hours
#     )

#     print(state)


    
# Test external input overide 
# for _ in range(steps_to_run):

#     if simulation.current_step == 22:
#         simulation.step({
#             "manual_mode":"battery_hold"
#                         })
#     else:
#         simulation.step()

# test_result = simulation.simulation_results_summary[22]

# print("Step:", test_result["Simulation_step"])
# print("Mode:", test_result["VPP_mode"])
# print("Battery Power:", test_result["Battery_power_kw"])
# next_result = simulation.simulation_results_summary[23]

# print("Step:", next_result["Simulation_step"])
# print("Mode:", next_result["VPP_mode"])
# print("Battery Power:", next_result["Battery_power_kw"])




# for _ in range(steps_to_run):

#     external_inputs = {}

#     if (
#         event_start_step
#         <= simulation.current_step
#         < event_end_step
#     ):
#         external_inputs = {
#             "network_support_request": True,
#             "requested_support_kw": 8.0
#         }

#     simulation.step(external_inputs)

House 1: PV=4.80 kW, Load=0.90 kW, Charge=5.00 kW, Discharge=0.00 kW, SOC=12.19 kWh
House 2: PV=2.40 kW, Load=1.10 kW, Charge=2.55 kW, Discharge=0.00 kW, SOC=5.44 kWh
House 3: PV=4.80 kW, Load=0.70 kW, Charge=-0.00 kW, Discharge=0.00 kW, SOC=0.00 kWh
House 4: PV=4.80 kW, Load=1.30 kW, Charge=4.00 kW, Discharge=0.00 kW, SOC=9.57 kWh
House 5: PV=0.00 kW, Load=1.00 kW, Charge=0.25 kW, Discharge=0.00 kW, SOC=1.74 kWh
VPP mode: self_consumption


{'House 1': 'charging', 'House 2': 'charging', 'House 3': 'no_battery', 'House 4': 'charging', 'House 5': 'minimum'}
